In [106]:
import requests
import os
import pandas as pd
import numpy as np
import holidays
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.model_selection import TimeSeriesSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.preprocessing import TargetEncoder
from sklearn.metrics import root_mean_squared_error
from sklearn.inspection import permutation_importance
import matplotlib.pyplot as plt

# Chargement des données et pré-traitement

In [107]:
def load_data(url):
    response = requests.get(url)

    #On recup les données et on les met sous forme csv
    if response.status_code // 100 == 2:
        data = response.json()
        availableBikeNumber= data.get("availableBikeNumber")
        values = availableBikeNumber.get("values")
        data_clean = []
        for value in values:
            bikeNumber = value[0]
            date = value[1]
            line = {"availableBikeNumber": bikeNumber, "date": date}
            data_clean.append(line)
        df = pd.DataFrame(data_clean)
        os.makedirs("data", exist_ok=True)
        df.to_csv("data/sample.csv", index=False)
        print("Données récupérées")

        return df

    else :
        print(f"Erreur : {response.status_code}")
        return None



In [108]:
def load_stations():
    url_stations = "https://gbfs.theta.fifteen.eu/gbfs/2.2/montpellier/en/station_information.json"
    response = requests.get(url_stations).json()

    # Extraction des données
    stations_list = response["data"]["stations"]
    df_locations = pd.DataFrame(stations_list)

    # Conservation des colonnes clés
    df_locations = df_locations[["station_id", "name", "lat", "lon"]]

    # Conversion en chaîne propre avec zéro initial (ex: 41 -> "041")
    df_locations["station_id"] = (
        df_locations["station_id"].astype(str).str.zfill(3)
    )

    # Sauvegarde pour Power BI
    os.makedirs("results", exist_ok=True)
    df_locations.to_csv(
        "results/stations.csv", index=False, encoding="utf-8-sig"
    )
    return df_locations

In [109]:
def load_all_stations_historical():
    url = (
        "https://portail-api-data.montpellier.fr/ngsi-ld/v1/temporal/entities"
        "?type=BikeHireDockingStation"
        "&format=temporalValues"
        "&timerel=after"
        "&timeAt=2025-12-31T23%3A59%3A59Z"
    )

    response = requests.get(url)

    # On accepte 200 (OK) ET 206 (Partial Content)
    if response.status_code in [200, 206]:
        entities = response.json()
        data_clean = []

        for entity in entities:
            full_id = entity.get("id", "")
            station_id = (
                full_id.split(":")[-1] if ":" in full_id else full_id
            )

            available_bike = entity.get("availableBikeNumber", {})
            values = available_bike.get("values", [])

            for bike_number, date in values:
                data_clean.append(
                    {
                        "station_id": station_id,
                        "availableBikeNumber": bike_number,
                        "date": date,
                    }
                )

        df = pd.DataFrame(data_clean)
        os.makedirs("data", exist_ok=True)
        df.to_csv("data/all_stations.csv", index=False)

        print(
            f"Données récupérées avec succès ! ({len(df)} lignes, statut {response.status_code})"
        )
        return df
    else:
        print(f"Erreur HTTP {response.status_code} : {response.text}")
        return None

In [110]:
def load_latest_stations():
    # URL instantanée : ne renvoie QUE l'état actuel de chaque station
    url = "https://portail-api-data.montpellier.fr/ngsi-ld/v1/entities?type=BikeHireDockingStation"

    response = requests.get(url)

    if response.status_code == 200:
        entities = response.json()
        data_clean = []

        for entity in entities:
            full_id = entity.get("id", "")
            station_id = (
                full_id.split(":")[-1] if ":" in full_id else full_id
            )

            # En temps réel, la donnée est directement accessible sous "value"
            available_bike = (
                entity.get("availableBikeNumber", {}).get("value", 0)
            )
            free_slots = entity.get("freeSlotNumber", {}).get("value", 0)
            status = entity.get("status", {}).get("value", "Working")
            date_obs = entity.get("observationDateTime", {}).get("value", "")

            data_clean.append(
                {
                    "station_id": station_id,
                    "availableBikeNumber": available_bike,
                    "freeSlotNumber": free_slots,
                    "status": status,
                }
            )

        df_latest = pd.DataFrame(data_clean)
        os.makedirs("data", exist_ok=True)
        df_latest.to_csv("data/latest_stations.csv", index=False)

        print(f"✅ Temps réel mis à jour : {len(df_latest)} stations.")
        return df_latest
    else:
        print(f"❌ Erreur HTTP {response.status_code}")
        return None

In [111]:
#df = pd.read_csv("data/sample.csv")

def clean_data(df):
    df["date"] = (pd.to_datetime(df["date"], utc=True).dt.tz_convert("Europe/Paris").dt.tz_localize(None))
    df["availableBikeNumber"] = df["availableBikeNumber"].astype(float)

    #Nettoyage et rajout de colonnes pour avoir + d'informations
    df["time"] = df["date"].dt.time
    df["hour"] = df["date"].dt.hour
    df["dayOfWeek"] = df["date"].dt.dayofweek
    df["isWeekend"] = df["date"].dt.dayofweek.isin([5,6]).astype(int)
    annees = [int(y) for y in df["date"].dt.year.unique() if pd.notna(y)]

    if annees:
        holidays_france = holidays.France(years=annees)
        df["isHolidays"] = df["date"].dt.date.isin(holidays_france).astype(int)
    else:
        df["isHolidays"] = 0

    return df

#pd.set_option('display.width', 1000)
#print(df.head())

In [112]:
def fusion(df_fusion1, df_fusion2, nom_fichier="donnees_fusionnees"):
    # 1. Copie pour ne pas modifier les DataFrames originaux hors de la fonction
    df1 = df_fusion1.copy()
    df2 = df_fusion2.copy()

    # 2. Uniformisation propre du type station_id
    df1["station_id"] = df1["station_id"].astype(str).str.strip()
    df2["station_id"] = df2["station_id"].astype(str).str.strip()

    # 3. Fusion
    df_result = df1.merge(df2, on="station_id", how="left")

    # 4. Création sécurisée du dossier results/ s'il n'existe pas
    os.makedirs("results", exist_ok=True)

    # 5. Sauvegarde
    chemin_fichier = f"results/{nom_fichier}.csv"
    df_result.to_csv(chemin_fichier, index=False)

    print(f"✅ Fichier sauvegardé sous : {chemin_fichier}")
    return df_result

# Entraînement du modèle

In [113]:

#Crée le dataframe où sont placés nos résultats
def create_results(X_test, y_test, predictions):
    df_resultats = X_test.copy()
    df_resultats["velosReels"] = y_test
    df_resultats["Erreur absolue"] = np.round(abs(y_test - predictions), 2)
    df_resultats["velosPredits"] = np.round(predictions, 0).astype(int)
    return df_resultats

#Entraîne le modèle
def evaluate_model(pipeline, X, y, n_splits=5):
    #kfold temporel
    tscv = TimeSeriesSplit(n_splits=n_splits)

    mae_scores = []
    rmse_scores = []
    liste_df_resultats = []

    #Entraînement des données et prédictions
    for fold, (train_index, test_index) in enumerate(tscv.split(X), 1):
        #Séparation des données
        X_train, X_test = X.iloc[train_index], X.iloc[test_index]
        y_train, y_test = y.iloc[train_index], y.iloc[test_index]

        #Entraînement du modèle sur données train
        pipeline.fit(X_train, y_train)

        #Prédiction des données test
        predictions = pipeline.predict(X_test)

        #Calcul des indicateurs de performance
        mae = mean_absolute_error(y_test, predictions)
        rmse = root_mean_squared_error(y_test, predictions)

        mae_scores.append(mae)
        rmse_scores.append(rmse)

        #On place les résultats dans une liste
        df_resultats = create_results(X_test, y_test, predictions)
        liste_df_resultats.append(df_resultats)

        print(f"Tour {fold} — MAE : {mae:.2f} | RMSE : {rmse:.2f}")

    #On regroupe tous les résultats
    df_resultats_finaux = pd.concat(liste_df_resultats, ignore_index=True)

    print("\n--- Résultats ---")
    print(f"MAE Moyenne : {np.mean(mae_scores):.2f}")
    print(f"RMSE Moyenne : {np.mean(rmse_scores):.2f}")

    return df_resultats_finaux

categorical_features = ["station_id"]
numeric_features = ["hour", "dayOfWeek", "isHolidays", "isWeekend"]

#Création du pipeline
columnTransformer = ColumnTransformer(
    transformers=[
        (
            "cat",
            TargetEncoder(smooth="auto", target_type="continuous"),
            categorical_features,
        ),
        (
            "num",
            "passthrough",
            numeric_features,
        ),
    ]
)

pipeline = Pipeline([("columnTransformer", columnTransformer), ("model", RandomForestRegressor(random_state=42, n_jobs=-1))])



# Main

In [114]:
#URL de l'historique des places Velomagg de la station 001
url = "https://portail-api-data.montpellier.fr/ngsi-ld/v1/temporal/entities/urn%3Angsi-ld%3Astation%3A001?format=temporalValues&timerel=after&timeAt=2025-12-31T23%3A59%3A59Z"


print("Chargement des données...")
df = load_all_stations_historical()
df_realtime = load_latest_stations()
df_stations = load_stations()
#df = load_data(url)

print("Nettoyage des données...")
df_clean = clean_data(df)

features = ["station_id", "hour", "dayOfWeek", "isHolidays", "isWeekend"]
X = df_clean[features]
y = df_clean["availableBikeNumber"]

print("Entraînement du modèle...")
df_resultats_finaux = evaluate_model(pipeline, X, y, n_splits=5)


#Importation csv pour PowerBI de nos résultats
os.makedirs("results", exist_ok=True)
df_resultats_finaux.to_csv("results/resultats.csv", index=False, encoding="utf-8-sig")
df_realtime.to_csv("results/realtime.csv", index=False, encoding="utf-8-sig")
print("Résultats enregistrés\n")

fusion(df_resultats_finaux, df_stations, "resultats_complets")
fusion(df_realtime, df_stations, "realtime_complet")

#Affichage de nos résultats
pd.set_option('display.width', 1000)
print(df_resultats_finaux.head())


Chargement des données...
Données récupérées avec succès ! (520000 lignes, statut 206)
✅ Temps réel mis à jour : 52 stations.
Nettoyage des données...
Entraînement du modèle...
Tour 1 — MAE : 2.53 | RMSE : 3.11
Tour 2 — MAE : 3.09 | RMSE : 4.02
Tour 3 — MAE : 5.70 | RMSE : 11.62
Tour 4 — MAE : 3.04 | RMSE : 4.22
Tour 5 — MAE : 4.68 | RMSE : 5.56

--- Résultats ---
MAE Moyenne : 3.81
RMSE Moyenne : 5.71
Résultats enregistrés

✅ Fichier sauvegardé sous : results/resultats_complets.csv
✅ Fichier sauvegardé sous : results/realtime_complet.csv
  station_id  hour  dayOfWeek  isHolidays  isWeekend  velosReels  Erreur absolue  velosPredits
0        041    16          0           0          0         2.0             0.0             2
1        041    16          0           0          0         2.0             0.0             2
2        041    16          0           0          0         2.0             0.0             2
3        041    16          0           0          0         2.0           

In [115]:

tscv = TimeSeriesSplit(n_splits=5)
splits = list(tscv.split(X))
train_index, test_index = splits[-1]  # dernier fold = celui sur lequel pipeline est actuellement fit

X_test = X.iloc[test_index]
y_test = y.iloc[test_index]

result = permutation_importance(
    pipeline, X_test, y_test, n_repeats=10, random_state=42, n_jobs=-1
)
importances_perm = pd.Series(
    result.importances_mean, index=X_test.columns
).sort_values(ascending=False)
print(importances_perm)

dayOfWeek     0.005247
station_id    0.004296
hour          0.001616
isWeekend    -0.000500
isHolidays   -0.009483
dtype: float64
